# Phase 5 — Financial + Operational specialists, orchestration, consensus

**Roadmap reference:** `05_Multi_Agent_Orchestration.ipynb`

**Scope of this notebook:** adds the Financial and Operational specialists
(reusing Phase 4's evidence-retrieval + verification engine, generalized in
`agents.py`), then orchestrates all three — sequential first, then
concurrent — and builds consensus into one `Assessment` per contract.

**Completion gate (from the roadmap):** one real agreement completes all
three assessments; a simulated agent failure yields a clear partial result;
a `CRITICAL` finding remains visible in `overall_risk` regardless of the
weighted average.

## 1. Load configuration and the manifest

Requires Phase 3's ChromaDB index (`chroma_contracts_db/`) — same as Phase 4,
each specialist retrieves evidence from it per-document, it does not re-embed.

In [1]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq.ingestion import load_manifest
from src.contractiq.agents import (
    FINANCIAL_RISK_TOPICS,
    OPERATIONAL_RISK_TOPICS,
    analyze_financial_risk,
    analyze_operational_risk,
)
from src.contractiq.orchestration import (
    run_sequential,
    run_parallel,
    build_consensus,
    save_assessment,
    run_phase5_sample,
)

manifest = load_manifest()
print(f"Loaded manifest: {len(manifest)} documents")
print(f"Financial risk topics probed per contract: {list(FINANCIAL_RISK_TOPICS)}")
print(f"Operational risk topics probed per contract: {list(OPERATIONAL_RISK_TOPICS)}")


Loaded manifest: 36 documents
Financial risk topics probed per contract: ['payment_terms', 'late_payment_penalties', 'pricing_structure', 'price_escalation', 'renewal_pricing', 'financial_exposure_cap']
Operational risk topics probed per contract: ['scope_of_work', 'acceptance_criteria', 'sla_commitments', 'sla_remedies', 'dependencies_obligations', 'performance_reporting']


## 2. Sequential orchestration on one contract

Roadmap-mandated first step: run Legal, Financial, and Operational one after
another, so if something's wrong it's obvious which specialist and which
attempt caused it, before adding concurrency.

In [2]:
import time

SAMPLE_DOC_ID = "MLA-001"

t0 = time.time()
sequential_results = run_sequential(SAMPLE_DOC_ID)
sequential_seconds = time.time() - t0

print(f"Sequential run on {SAMPLE_DOC_ID}: {sequential_seconds:.1f}s")
for dimension, result in sequential_results.items():
    print(f"  {dimension.value}: success={result.success}, {len(result.findings)} finding(s), "
          f"retry_count={result.retry_count}")


Sequential run on MLA-001: 36.8s
  legal: success=True, 7 finding(s), retry_count=0
  financial: success=True, 4 finding(s), retry_count=0
  operational: success=True, 5 finding(s), retry_count=0


## 3. Parallel orchestration — same contract, measured against sequential

Added only after the sequential path above was confirmed correct. Same three
specialists, same retry logic, run concurrently via a thread pool (I/O-bound
OpenAI calls, so threads are enough — no multiprocessing needed). The timing
is measured, not assumed, on the same document as the sequential run above.

In [3]:
t0 = time.time()
parallel_results = run_parallel(SAMPLE_DOC_ID)
parallel_seconds = time.time() - t0

print(f"Parallel run on {SAMPLE_DOC_ID}: {parallel_seconds:.1f}s")
for dimension, result in parallel_results.items():
    print(f"  {dimension.value}: success={result.success}, {len(result.findings)} finding(s), "
          f"retry_count={result.retry_count}")

print(f"\nSequential: {sequential_seconds:.1f}s  |  Parallel: {parallel_seconds:.1f}s  "
      f"|  Speedup: {sequential_seconds / parallel_seconds:.1f}x")


Parallel run on MLA-001: 14.5s
  financial: success=True, 4 finding(s), retry_count=0
  operational: success=True, 5 finding(s), retry_count=0
  legal: success=True, 7 finding(s), retry_count=0

Sequential: 36.8s  |  Parallel: 14.5s  |  Speedup: 2.5x


## 4. Consensus — merge findings into one Assessment

`build_consensus()` merges the three dimension results, deduplicates only
genuinely equivalent findings (same dimension + same title + shared
evidence), and applies the documented risk policy (Legal 40% / Financial
35% / Operational 25%) — **except** a `CRITICAL` finding anywhere always
forces `overall_risk = CRITICAL`, regardless of the weighted average. This
override is verified separately below with a synthetic case, since no real
document in this corpus has produced a genuine CRITICAL finding yet.

In [4]:
assessment = build_consensus(SAMPLE_DOC_ID, sequential_results)
save_assessment(assessment)

print(f"doc_id: {assessment.doc_id}")
print(f"overall_risk: {assessment.overall_risk.value}")
print(f"unresolved_issues: {assessment.unresolved_issues}")
print(f"consensus_actions ({len(assessment.consensus_actions)}):")
for action in assessment.consensus_actions:
    print(f"  - {action}")
print(f"model_version={assessment.model_version}, policy_version={assessment.policy_version}, run_id={assessment.run_id}")


doc_id: MLA-001
overall_risk: HIGH
unresolved_issues: []
consensus_actions (7):
  - Review: [legal] Limitation of Liability Found
  - Review: [legal] Indemnification Obligations
  - Review: [legal] Intellectual Property Ownership
  - Review: [legal] Governing Law and Dispute Resolution
  - Review: [operational] Scope of Work Defined
  - Review: [operational] Service Level Commitments
  - Review: [operational] Service Performance Measurement and Reporting
model_version=gpt-4o-mini, policy_version=risk-policy-v1, run_id=e2a8a362d3ea


### Completion-gate check: a CRITICAL finding is never averaged away

Synthetic case: Legal has only LOW findings, Operational fails outright
(simulated), and Financial has one CRITICAL finding. A naive weighted
average would land near LOW/MEDIUM — `overall_risk` must still be
`CRITICAL`, and the failed Operational dimension must show up in
`unresolved_issues`.

In [5]:
from src.contractiq.models import DimensionResult, Finding, RiskDimension, Severity, VerificationStatus

synthetic_legal = DimensionResult(dimension=RiskDimension.LEGAL, success=True, findings=[
    Finding(finding_id="legal-x-1", dimension=RiskDimension.LEGAL, severity=Severity.LOW,
            title="Minor clause", rationale="synthetic test finding", confidence=0.9,
            verification_status=VerificationStatus.VERIFIED),
])
synthetic_financial = DimensionResult(dimension=RiskDimension.FINANCIAL, success=True, findings=[
    Finding(finding_id="financial-x-1", dimension=RiskDimension.FINANCIAL, severity=Severity.CRITICAL,
            title="Uncapped financial exposure", rationale="synthetic test finding", confidence=0.9,
            verification_status=VerificationStatus.VERIFIED),
])
synthetic_operational = DimensionResult(dimension=RiskDimension.OPERATIONAL, success=False, findings=[],
                                         error="simulated timeout")

synthetic_assessment = build_consensus("TEST-SYNTHETIC", {
    RiskDimension.LEGAL: synthetic_legal,
    RiskDimension.FINANCIAL: synthetic_financial,
    RiskDimension.OPERATIONAL: synthetic_operational,
})

print(f"overall_risk: {synthetic_assessment.overall_risk.value} (expect CRITICAL)")
print(f"unresolved_issues: {synthetic_assessment.unresolved_issues}")
assert synthetic_assessment.overall_risk == Severity.CRITICAL
assert len(synthetic_assessment.unresolved_issues) == 1
print("PASS — CRITICAL override and unresolved-issue tracking both hold.")


overall_risk: CRITICAL (expect CRITICAL)
unresolved_issues: ['operational: analysis did not succeed after 1 attempt(s) — simulated timeout']
PASS — CRITICAL override and unresolved-issue tracking both hold.


## 5. Scale to the 15 source-verified documents

Same 15 documents used to scale Phase 4 (MLA-001..006, FIN-001/002/004,
COM-001/002, NEG-003, GOV-004, OPS-003/004). `run_phase5_sample()` runs the
full three-specialist + consensus pipeline (parallel) for each, saves every
`Assessment`, and prints a one-line summary per doc.

In [6]:
BATCH_DOC_IDS = [
    "MLA-001", "MLA-002", "MLA-003", "MLA-004", "MLA-005", "MLA-006",
    "FIN-001", "FIN-002", "FIN-004",
    "COM-001", "COM-002",
    "NEG-003",
    "GOV-004",
    "OPS-003", "OPS-004",
]
batch_assessments = run_phase5_sample(BATCH_DOC_IDS, parallel=True)

from collections import Counter
risk_distribution = Counter(a.overall_risk.value for a in batch_assessments.values())
print(f"\noverall_risk distribution across {len(batch_assessments)} docs: {dict(risk_distribution)}")


MLA-001: overall_risk=HIGH, 16 finding(s) across 3 dimensions, 0 unresolved issue(s), 6 consensus action(s)
MLA-002: overall_risk=HIGH, 13 finding(s) across 3 dimensions, 0 unresolved issue(s), 5 consensus action(s)
MLA-003: overall_risk=HIGH, 7 finding(s) across 3 dimensions, 0 unresolved issue(s), 3 consensus action(s)
MLA-004: overall_risk=HIGH, 6 finding(s) across 3 dimensions, 0 unresolved issue(s), 3 consensus action(s)
MLA-005: overall_risk=HIGH, 10 finding(s) across 3 dimensions, 0 unresolved issue(s), 3 consensus action(s)
MLA-006: overall_risk=MEDIUM, 7 finding(s) across 3 dimensions, 0 unresolved issue(s), 1 consensus action(s)
FIN-001: overall_risk=MEDIUM, 8 finding(s) across 3 dimensions, 0 unresolved issue(s), 0 consensus action(s)
FIN-002: overall_risk=MEDIUM, 9 finding(s) across 3 dimensions, 0 unresolved issue(s), 1 consensus action(s)
FIN-004: overall_risk=MEDIUM, 7 finding(s) across 3 dimensions, 0 unresolved issue(s), 0 consensus action(s)
COM-001: overall_risk=MEDI

## Honest status of Phase 5 at this point

- **Built:** Financial specialist (`analyze_financial_risk`) and Operational
  specialist (`analyze_operational_risk`), both reusing Phase 4's
  evidence-retrieval + verification engine (generalized into
  `analyze_dimension()` in `agents.py` — no duplicated logic per dimension).
  `run_sequential()` / `run_parallel()` orchestrate all three with bounded
  retries; `build_consensus()` merges findings, dedupes only genuinely
  equivalent ones, and applies the documented risk policy with a verified
  CRITICAL override.
- **Verified, not assumed:** parallel execution measured faster than
  sequential on the same document (see Section 3); the CRITICAL-override
  and unresolved-issue behaviors are checked with an explicit synthetic
  case (Section 4), not just asserted in a docstring.
- **Known limitation:** Financial and Operational topics do **not** yet have
  a hand-verified labeled test set the way Phase 3 built one for retrieval —
  their `failed_verification` catches (e.g. FIN-001's pricing_structure
  finding, which stitched together non-adjacent sentences instead of
  quoting verbatim) are the only quality signal so far. A proper
  precision/recall evaluation for these two specialists is Phase 7's job
  (`07_Evaluation_and_Observability.ipynb`), not built here.
- **Not built yet (Phase 6+):** knowledge graph relationships, the
  frozen evaluation set across all three specialists, and the Gradio UI.
- **Not real yet:** anything in `PWC_CONTRACTIQ_PROJECT_IK/` (the earlier
  course-assignment project — `ContractIntelligenceSystem`,
  `ContractRiskOrchestrator`, `ContractKnowledgeGraph`). Separate project,
  not wired into this capstone rebuild.